In [ ]:
import os
from tqdm import tqdm

# Third parties
import numpy as np
import keras

# Import from other modules
# from tagger.data.tools import load_data, to_ML, select_events
# from tagger.model.common import fromFolder
# from tagger.plot.basic import basic, plot_event_ROC, plot_PCA, plot_latent,plot_nontrained_event_ROC,plot_latent_vs_variable,plot_output_scores
# import tagger.plot.style as style 

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
# import mplhep as hep

from config import PROCESS_INFO, JET_INFO
process_info = PROCESS_INFO.copy()
process_info['QCD'] = {'path': None, 'fraction': 0, 'class': 1, 'label': 'QCD', 'type': 'SM', 'train': True}  # Add a placeholder for QCD to keep it in the process_info dictionary
process_info = {k: v for k, v in process_info.items() if 'QCD_' not in k}  # Exclude QCD samples for embedding analysis

jet_info = JET_INFO

outdir = os.path.join("plots_inputs")
os.makedirs(outdir, exist_ok=True)

print("Samples:", process_info.keys())

In [ ]:
sample = 'QCD_Pt15To3000_PU200'
# sample = 'MinBias_PU200'
# sample = 'TT_PU200'
# sample = 'SUEP'
# sample = 'SVJ_250'
# sample = 'SVJ_500'
# sample = 'SMJ_cascadeA'
# sample = 'SMJ_cascadeC'
data = np.load(os.path.join('../data_numpy', f'{sample}.npz'), allow_pickle=True)
all_event_id = data['event_id']
all_event_class = data['event_class']
all_jet_class = data['jet_class']
all_jet_features = data['jet_features']
all_nn_inputs = data['nn_inputs']
del data

all_jet_pt = all_jet_features[:, 1]
all_jet_eta = all_jet_features[:, 2]
all_jet_ncand = all_jet_features[:, 4]
all_jet_pt_phys = all_jet_features[:, 5]
all_jet_eta_phys = all_jet_features[:, 6]
all_jet_reject = all_jet_features[:, 15]
all_target_pt_phys = all_jet_features[:, 14]

valid_jet_mask = (
    (all_jet_pt_phys > 15.0) & (all_jet_eta_phys < 2.4) & (all_jet_reject == 0) & (all_target_pt_phys > 5.0)
)

event_id = all_event_id[valid_jet_mask]
event_class = all_event_class[valid_jet_mask]
jet_class = all_jet_class[valid_jet_mask]
jet_features = all_jet_features[valid_jet_mask]
nn_inputs = all_nn_inputs[valid_jet_mask]
jet_pt = all_jet_pt[valid_jet_mask]
jet_eta = all_jet_eta[valid_jet_mask]
jet_ncand = all_jet_ncand[valid_jet_mask]
jet_pt_phys = all_jet_pt_phys[valid_jet_mask]
jet_eta_phys = all_jet_eta_phys[valid_jet_mask]

print("Total jets:", len(all_event_id))
print("Valid jets:", len(event_id))


In [ ]:
feature_list = [
    "pt", "pt_rel", "pt_log", 
    "deta", "dphi", "mass", 
    "isPhoton", "isElectronPlus", "isElectronMinus", "isMuonPlus", "isMuonMinus", "isNeutralHadron", "isChargedHadronPlus", "isChargedHadronMinus", 
    "z0", "dxy", 
    "isfilled", "puppiweight", "emid", "quality",
]
nfeatures = len(feature_list)

features_to_plot = [
    "pt", "pt_rel", "pt_log", 
    "deta", "dphi", "mass", 
    "isPhoton", "isElectronPlus", "isElectronMinus", "isMuonPlus", "isMuonMinus", "isNeutralHadron", "isChargedHadronPlus", "isChargedHadronMinus", 
    "z0", "dxy", 
    "isfilled", 
    "puppiweight", "emid", "quality",
]
nfeatures_to_plot = len(features_to_plot)

fig, ax = plt.subplots(nfeatures_to_plot, 4, figsize=(16, 40))

for i in range(nfeatures_to_plot):
    idx = feature_list.index(features_to_plot[i])
    print(f"Plotting feature {features_to_plot[i]} (index {idx})")

    xrange = (np.percentile(nn_inputs[:, :, idx], 0.1), np.percentile(nn_inputs[:, :, idx], 99.9))
    ax[i, 0].hist(nn_inputs[:, 0, idx], bins=50, range=xrange, histtype='step', color='black', alpha=1)
    ax[i, 1].hist(nn_inputs[:, 1, idx], bins=50, range=xrange, histtype='step', color='black', alpha=1)
    ax[i, 2].hist(nn_inputs[:, 2, idx], bins=50, range=xrange, histtype='step', color='black', alpha=1)
    ax[i, 3].hist(nn_inputs[:, 3, idx], bins=50, range=xrange, histtype='step', color='black', alpha=1)
    
    for j in range(4):
        ax[i, j].set_xlim(xrange)
        ax[i, j].set_yticks([])
        ax[i, j].set_yscale('log')
        if i == 0:
            ax[i, j].set_title(f'Constituent {j+1}')
        if j == 0:
            ax[i, j].set_ylabel(features_to_plot[i])

plt.tight_layout()
plt.savefig(os.path.join(outdir, f"input_features_{sample.lower()}.pdf"))
plt.show()

In [ ]:
with open(os.path.join(outdir, f"feature_statistics_{sample.lower()}.txt"), 'w') as f:
    for idx, feature in enumerate(feature_list):
        stats = (
            f"Feature {feature:<25} "
            f"(index {idx:2d}): "
            f"min={np.min(nn_inputs[:, :, idx]):10.3f}, "
            f"max={np.max(nn_inputs[:, :, idx]):10.3f}, "
            f"mean={np.mean(nn_inputs[:, :, idx]):10.3f}, "
            f"std={np.std(nn_inputs[:, :, idx]):10.3f}"
        )

        print(stats)
        f.write(stats + "\n")

In [ ]:
for f in ['mass', 'isfilled', 'emid', 'quality', 'deta', 'dphi', 'z0', 'dxy', 'puppiweight']:
    idx = feature_list.index(f)

    plt.figure(figsize=(6, 4))

    plt.hist(
        [nn_inputs[:, 0, idx], nn_inputs[:, 1, idx], nn_inputs[:, 2, idx], nn_inputs[:, 3, idx],][::-1],
        bins=100,
        stacked=True,
        histtype='bar',
        alpha=0.5,
        label=['Constituent 1', 'Constituent 2', 'Constituent 3', 'Constituent 4'][::-1]
    )
    # plt.yscale('log')
    plt.legend()
    plt.title(f)
    plt.show()